In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
pd.options.display.max_colwidth = 300

In [ ]:
data_path = '/Volumes/ja2/vegan/vegan_parser/scraping/ewg_data/'

In [ ]:
column_names = ['datetime_pulled',
               'search_page_num',
               'chemical_name',
               'score_url',
               'data_availability',
               'chemical_concerns',
               'chemical_functions',
               'chemical_about',
               'chemical_synonyms']

In [ ]:
filepaths = [os.path.join(data_path,f) for f in os.listdir(data_path) if f.startswith('chemical_details')]

In [ ]:
len(filepaths)

In [ ]:
df = pd.concat([pd.read_csv(f, names=column_names) for f in filepaths], axis=0)

In [ ]:
df.shape

In [ ]:
df[df['chemical_name']=='SODIUM RAPESEEDATE']

In [ ]:
df[df['chemical_name']=='SODIUM RAPESEEDATE']['chemical_synonyms'].values

In [ ]:
# Oh, it's not just a list, there's an "and" in there... but if the names are always capitalized, that's a way to deal with it.
# Let's look up the alternative names


In [ ]:
df[df['chemical_name']=='FATTY ACIDS']

In [ ]:
# OH COME ON - these data are different!
# Because Sodium Rapeseedate is one of many Fatty Acids out there.

In [ ]:
df[df['chemical_name']=='RAPESEED OIL']

In [ ]:
# Okay, but it looks like direct synoyms are not in the list.

### A few things to do:
- duplicate rows for all the chemical synonyms, so we can have all the names in a column to look up
- translate the score_url to actual scores


In [ ]:
# Translating score_url to actual scores.

df['score_url'].unique()

The scores that have a range means it depends on usage.
But, to be fair, water has a value of 1, but if you inhale too much water, or drink way too much of it, you can die. So, we should take these with a grain of salt

In [ ]:
df[df['chemical_name']=='WATER']

In [ ]:
df.iloc[0,3]

In [ ]:
# Let's write out the min and max scores to their own columns, and average them to get an average score
# we'll need a function to read these out

In [ ]:
def extract_max(string):
    score_range = [int(s) for s in string if s.isdigit()]
    return score_range[0]

def extract_min(string):
    score_range = [int(s) for s in string if s.isdigit()]
    return score_range[1]

def extract_mean(string):
    score_range = [int(s) for s in string if s.isdigit()]
    return np.mean(score_range)

In [ ]:
'111'.isdigit()

In [ ]:
string = 'https://www.ewg.org/skindeep/squircle/show.svg?score=9&score_min=4'

In [ ]:
[int(s) for s in string if s.isdigit()]

In [ ]:
np.mean([1, 1])

In [ ]:
df['min_score'] = df['score_url'].apply(lambda x: extract_min(x))
df['max_score'] = df['score_url'].apply(lambda x: extract_max(x))
df['mean_score'] = df['score_url'].apply(lambda x: extract_mean(x))

In [ ]:
pd.options.display.max_colwidth = 250

In [ ]:
df['chemical_synonyms'].sample(40)

# After looking at a bunch of entries, it looks like each ingredient is split by ',' OR ";" OR "and".  (And sometimes, both.)
# Also noticed a few instances of spaces where there shouldn't be (e.g. "DIS ODIUM" instead of "DISODIUM"), but we can't do much about that. 
# Also, some commas are included where they are part of the name of the chemical: "1,2,3-PROPANETRIOL, 1,2,3-TRIHYDROXYPROPANE, 1,2,3PROPANETRIOL, ..."
Hopefully, we can do some fuzzy matching and catch any missing text
We'll have to ignore any numbers

In [ ]:
not "1".isdigit()

In [ ]:
def extract_synonymns_list(str1):
#     print(str1)
    if pd.isna(str1):
        return []
    
    split_and_list = str1.split('and')
    split_and_comma_list_in_list = [string.split(',') for string in split_and_list]
    split_and_comma_semi_list_in_list = [item.strip().split(';') for sublist in split_and_comma_list_in_list for item in sublist]

    intermediate_list = [item for sublist in split_and_comma_semi_list_in_list for item in sublist if item!='']
   
    # also need to filter out numbers in the ingredients due to odd splitting
    
    return [item for item in intermediate_list if not item.isdigit()]
    

In [ ]:
str1 = 'PEG-75 MEADOWFOAM OIL, POLYETHYLENE GLYCOL 4000 MEADOWFOAM OIL, and POLYOXYETHYLENE (75) MEADOWFOAM OIL'
str2 = 'PEG-75 MEADOWFOAM OIL, POLYETHYLENE GLYCOL 4000 MEADOWFOAM OIL and POLYOXYETHYLENE (75) MEADOWFOAM OIL'
str3 = 'PEG-75 MEADOWFOAM OIL, POLYETHYLENE GLYCOL 4000 MEADOWFOAM OIL; and POLYOXYETHYLENE (75) MEADOWFOAM OIL'
str4 = '1, 2, 3 SOMETHING OR OTHER, WATER'


In [ ]:
extract_synonymns_list(str1)

In [ ]:
extract_synonymns_list(str2)

In [ ]:
extract_synonymns_list(str3)

In [ ]:
extract_synonymns_list(str4)

### Okay, that looks about right... 
I don't see any way out of iterating through the rows to deal with dupes.
We'll make a new dataframe and append to that.
I'm not going to bother including a value for the synonym'ed rows



In [ ]:
syn_col_names = list(df.columns)+['is_syn']
syn_col_names

In [ ]:
#

In [ ]:
print(datetime.now())

In [ ]:
df_test = df.head(20)

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.tail()

In [ ]:
# Note - it looks like the synonyms list in the scraped data includes the main chemical.
# So we don't have to worry about bringing that along

In [ ]:
# OH man this is going to be SLOW AF. But, I don't care if it takes a long time (like, under an hour)

df_wsyns = pd.DataFrame([],columns=syn_col_names)  # DataFrame WithSYNonymS
print(datetime.now())

for index, row in df.iterrows():
    
    if index%100==0:
        print(f'index: {index} of {len(df)} rows')
    
    syn_list = extract_synonymns_list(row['chemical_synonyms'])
    
    if syn_list!=[]:

        # append the original
        row['is_syn'] = False
        df_wsyns = df_wsyns.append(row, ignore_index=True)
                        
        for syn in syn_list:
            row['chemical_name'] = syn        
            row['is_syn'] = True
            if syn not in df_wsyns['chemical_name'].values:
                df_wsyns = df_wsyns.append(row, ignore_index=True)

    else:
        row['is_syn'] = False
        df_wsyns = df_wsyns.append(row, ignore_index=True)
    
print(datetime.now())
df_wsyns.to_csv('dataframe_with_syns.csv', index=False)
            

In [ ]:
df_wsyns.tail()

In [ ]:
# Took <15 min, that's fine.
# 34230 entries, 20MB. 

In [ ]:
!tail dataframe_with_syns.csv